In [1]:
import pandas as pd
import numpy as np

# This mimics what yf.download("^NSEI") actually returns - MultiIndex columns
columns = pd.MultiIndex.from_tuples([
    ('Close', '^NSEI'),
    ('High', '^NSEI'),
    ('Low', '^NSEI'),
    ('Volume', '^NSEI')
])

data = {
    ('Close', '^NSEI'): [24000, 24050, 24100, 24080, 24150, 24200, 24250, 24280, 24320, 24350,
                          24400, 24450, 24480, 24500, 24550, 24580, 24600, 24650, 24700, 24750],
    ('High', '^NSEI'): [24050, 24100, 24150, 24120, 24200, 24250, 24300, 24330, 24370, 24400,
                        24450, 24500, 24530, 24550, 24600, 24630, 24650, 24700, 24750, 24800],
    ('Low', '^NSEI'): [23950, 24000, 24050, 24030, 24100, 24150, 24200, 24230, 24270, 24300,
                       24350, 24400, 24430, 24450, 24500, 24530, 24550, 24600, 24650, 24700],
    ('Volume', '^NSEI'): [100, 110, 120, 95, 130, 140, 150, 145, 155, 160,
                          170, 180, 175, 185, 190, 195, 200, 210, 220, 230]
}

df = pd.DataFrame(data)

print("🔴 UNFLATTENED DataFrame (MultiIndex columns):")
print(df.head())
print("\nColumn structure:")
print(df.columns)



🔴 UNFLATTENED DataFrame (MultiIndex columns):
   Close   High    Low Volume
   ^NSEI  ^NSEI  ^NSEI  ^NSEI
0  24000  24050  23950    100
1  24050  24100  24000    110
2  24100  24150  24050    120
3  24080  24120  24030     95
4  24150  24200  24100    130

Column structure:
MultiIndex([( 'Close', '^NSEI'),
            (  'High', '^NSEI'),
            (   'Low', '^NSEI'),
            ('Volume', '^NSEI')],
           )


In [6]:
def calculate_rsi(series, period=14):
    """
    Calculate Relative Strength Index (RSI)
    """
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()

    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))

    return rsi

In [11]:
# Try to calculate RSI on unflattened data
print("Type of df['Close']:", type(df['Close']))
print("\ndf['Close'] shape:", df['Close'].shape)

try:
    df['RSI'] = calculate_rsi(df['Close'], period=14)
    print("\n✅ RSI calculation succeeded")
    print("\n📊 FULL RSI column (notice NaN at start, values at end):")
    print(df[['Close', 'RSI']])  # Show ALL rows, not just head()
    print(f"\nNon-NaN RSI values: {df['RSI'].notna().sum()} out of {len(df)}")
except Exception as e:
    print(f"\n❌ RSI calculation failed: {type(e).__name__}: {e}")

Type of df['Close']: <class 'pandas.core.series.Series'>

df['Close'] shape: (20,)

✅ RSI calculation succeeded

📊 FULL RSI column (notice NaN at start, values at end):
    Close         RSI
0   24000         NaN
1   24050         NaN
2   24100         NaN
3   24080         NaN
4   24150         NaN
5   24200         NaN
6   24250         NaN
7   24280         NaN
8   24320         NaN
9   24350         NaN
10  24400         NaN
11  24450         NaN
12  24480         NaN
13  24500   96.296296
14  24550   96.610169
15  24580   96.491228
16  24600   96.296296
17  24650  100.000000
18  24700  100.000000
19  24750  100.000000

Non-NaN RSI values: 7 out of 20


In [12]:
# Show the issue created by adding RSI to unflattened data
print("🔴 PROBLEM: Mixed column structure after adding RSI")
print("\nColumn structure:")
print(df.columns)
print(f"\nColumn types: {type(df.columns)}")

print("\n" + "="*60)
print("ISSUE 1: Trying to access 'Close' now:")
print("="*60)
try:
    close_data = df['Close']
    print(f"Type: {type(close_data)}")
    print(f"Shape: {close_data.shape}")
    print(close_data.head())
except Exception as e:
    print(f"❌ Error: {e}")

print("\n" + "="*60)
print("ISSUE 2: Trying to select multiple columns:")
print("="*60)
try:
    subset = df[['Close', 'High', 'RSI']]
    print(subset.head())
except Exception as e:
    print(f"❌ Error: {type(e).__name__}: {e}")

print("\n" + "="*60)
print("ISSUE 3: Column name confusion:")
print("="*60)
print("'Close' is actually:", df.columns[df.columns.get_loc('Close')])
print("'RSI' is actually:", 'RSI')
print("\nNotice: 'Close' is ('Close', '^NSEI') but 'RSI' is just 'RSI'")

🔴 PROBLEM: Mixed column structure after adding RSI

Column structure:
Index(['Close', 'High', 'Low', 'Volume', 'RSI'], dtype='object')

Column types: <class 'pandas.core.indexes.base.Index'>

ISSUE 1: Trying to access 'Close' now:
Type: <class 'pandas.core.series.Series'>
Shape: (20,)
0    24000
1    24050
2    24100
3    24080
4    24150
Name: Close, dtype: int64

ISSUE 2: Trying to select multiple columns:
   Close   High  RSI
0  24000  24050  NaN
1  24050  24100  NaN
2  24100  24150  NaN
3  24080  24120  NaN
4  24150  24200  NaN

ISSUE 3: Column name confusion:
'Close' is actually: Close
'RSI' is actually: RSI

Notice: 'Close' is ('Close', '^NSEI') but 'RSI' is just 'RSI'


In [13]:
print("\n" + "="*60)
print("THE REAL COLUMN STRUCTURE IN df:")
print("="*60)
print("Full df.columns:")
for i, col in enumerate(df.columns):
    print(f"  [{i}] {repr(col)} - type: {type(col).__name__}")

print(f"\nTotal columns: {len(df.columns)}")
print(f"MultiIndex? {isinstance(df.columns, pd.MultiIndex)}")


THE REAL COLUMN STRUCTURE IN df:
Full df.columns:
  [0] 'Close' - type: str
  [1] 'High' - type: str
  [2] 'Low' - type: str
  [3] 'Volume' - type: str
  [4] 'RSI' - type: str

Total columns: 5
MultiIndex? False
